# 04b — LangGraph Workflow

## Objective

Build an explicit stateful workflow using LangGraph.

Workflow:

START
  ↓
Router
  ↓
RAG Node OR Tool Node
  ↓
Responder
  ↓
END

We will also learn:

- Graph state
- Nodes
- Edges
- Conditional routing
- Checkpointers
- Human approval interrupts
- Graph visualization

In [1]:
%pip install -U langgraph

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip show langgraph

Name: langgraph
Version: 1.2.12
Summary: Building stateful, multi-actor applications with LLMs
Home-page: https://docs.langchain.com/oss/python/langgraph/overview
Author: 
Author-email: 
License-Expression: MIT
Location: D:\arun\ai-learning\.venv\Lib\site-packages
Requires: langchain-core, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk, pydantic, xxhash
Required-by: langchain
Note: you may need to restart the kernel to use updated packages.


In [3]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

print("LangGraph imports successful.")

LangGraph imports successful.


## Graph State

LangGraph workflows operate on shared state.

Our state contains:

- question — user's request
- route — selected workflow
- context — retrieved document information
- tool_result — result from a tool
- answer — final response

In [4]:
class AgentState(TypedDict):
    question: str
    route: str
    context: str
    tool_result: str
    answer: str

## Reuse Existing Components

The goal is not to rebuild the LLM and tools.

We will reuse the concepts from 04a:

- Groq LLM
- calculator
- document lookup
- mock database

In [5]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool

import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set.")

MODEL = "openai/gpt-oss-20b"

model = ChatOpenAI(
    model=MODEL,
    temperature=0,
    api_key=OPENAI_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

print("Model configured:", MODEL)

Model configured: openai/gpt-oss-20b


## Router Node

The router decides which path should handle the request.

For this learning workflow:

- Questions about documents/RAG → `rag`
- Requests requiring a tool → `tool`

The router updates the graph state.

In [6]:
def router(state: AgentState):
    question = state["question"].lower()

    rag_keywords = [
        "docuchat",
        "rag",
        "document",
        "embedding",
        "retrieval"
    ]

    tool_keywords = [
        "calculate",
        "multiply",
        "divide",
        "database",
        "users",
        "documents"
    ]

    if any(keyword in question for keyword in rag_keywords):
        route = "rag"

    elif any(keyword in question for keyword in tool_keywords):
        route = "tool"

    else:
        route = "rag"

    return {
        "route": route
    }

## RAG Node

For this notebook, the RAG node will use our simple
`document_lookup` tool as the retrieval layer.

This is a workflow demonstration, not the production DocuChat
retrieval implementation.

In [7]:
def rag_node(state: AgentState):
    question = state["question"]

    context = document_lookup.invoke(
        {
            "query": question
        }
    )

    return {
        "context": context
    }

## Tool Node

The tool node executes an appropriate tool based on the request.

For this basic workflow we will support:

- calculator
- mock database query

In [8]:
def tool_node(state: AgentState):
    question = state["question"].lower()

    if "calculate" in question:
        expression = (
            question
            .replace("calculate", "")
            .strip()
        )

        result = calculator.invoke(
            {
                "expression": expression
            }
        )

    elif "users" in question:
        result = mock_db_query.invoke(
            {
                "table": "users",
                "limit": 5
            }
        )

    elif "database" in question:
        result = mock_db_query.invoke(
            {
                "table": "documents",
                "limit": 5
            }
        )

    else:
        result = "No suitable tool found."

    return {
        "tool_result": result
    }

## Responder Node

The responder generates the final answer from the information
produced by either:

- the RAG node
- the Tool node

In [9]:
def responder(state: AgentState):
    question = state["question"]
    route = state["route"]

    if route == "rag":
        information = state.get("context", "")

        prompt = f"""
Answer the user's question using only the provided context.

Question:
{question}

Context:
{information}

If the context does not contain enough information, say that
the information was not found in the available documents.
"""

    else:
        information = state.get("tool_result", "")

        prompt = f"""
Answer the user's question using the tool result.

Question:
{question}

Tool result:
{information}
"""

    response = model.invoke(prompt)

    return {
        "answer": response.content
    }

## Conditional Routing

After the router runs, LangGraph needs to know which node should
execute next.

The routing function reads `state["route"]`.

In [10]:
def route_function(state: AgentState):
    return state["route"]

## Build the LangGraph Workflow

We now connect the nodes.

Workflow:

START
  ↓
router
  ↓
conditional route
  ├── rag
  └── tool
       ↓
   responder
       ↓
      END

In [11]:
builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("rag", rag_node)
builder.add_node("tool", tool_node)
builder.add_node("responder", responder)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_function,
    {
        "rag": "rag",
        "tool": "tool"
    }
)

builder.add_edge("rag", "responder")
builder.add_edge("tool", "responder")

builder.add_edge("responder", END)

print("Graph structure created successfully.")

Graph structure created successfully.


In [12]:
graph = builder.compile()

print("LangGraph compiled successfully.")

LangGraph compiled successfully.


In [13]:
DOCUMENTS = {
    "docuchat": """
    DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.
    """,

    "rag": """
    Retrieval-Augmented Generation combines document retrieval
    with language model generation. Relevant context is retrieved
    before the answer is generated.
    """,

    "agents": """
    An AI agent can decide which action to take, use tools,
    observe results, and continue until the task is complete.
    """
}

print("Documents loaded:", list(DOCUMENTS.keys()))

Documents loaded: ['docuchat', 'rag', 'agents']


In [14]:
@tool
def document_lookup(query: str) -> str:
    """Search the document knowledge base using a keyword query."""

    query_lower = query.lower()
    matches = []

    for name, content in DOCUMENTS.items():
        if name in query_lower:
            matches.append(content.strip())

    if not matches:
        return "No relevant document found."

    return "\n\n".join(matches)


In [15]:
result = document_lookup.invoke(
    {"query": "Tell me about DocuChat"}
)

print(result)

DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.


In [16]:
def rag_node(state: AgentState):
    question = state["question"]

    context = document_lookup.invoke(
        {
            "query": question
        }
    )

    return {
        "context": context
    }

In [17]:
result = graph.invoke(
    {
        "question": "What is DocuChat?",
        "route": "",
        "context": "",
        "tool_result": "",
        "answer": ""
    }
)

print("Route:", result["route"])
print("Context:", result["context"])
print("Answer:", result["answer"])

Route: rag
Context: DocuChat is a RAG-powered chat application.
    It uses PostgreSQL with pgvector to store document embeddings.
    Relevant document chunks are retrieved using semantic similarity.
Answer: DocuChat is a RAG‑powered chat application that uses PostgreSQL with pgvector to store document embeddings and retrieves relevant document chunks through semantic similarity.


In [19]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""

    allowed_chars = "0123456789+-*/(). "

    if not all(char in allowed_chars for char in expression):
        return "Error: expression contains unsupported characters."

    try:
        result = eval(
            expression,
            {"__builtins__": {}},
            {}
        )

        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [20]:
print(
    calculator.invoke(
        {"expression": "25 * 47"}
    )
)

1175


In [21]:
MOCK_DB = {
    "users": [
        {
            "id": 1,
            "name": "Arun",
            "email": "arun@example.com"
        },
        {
            "id": 2,
            "name": "Raj",
            "email": "raj@example.com"
        }
    ],

    "documents": [
        {
            "id": 1,
            "filename": "docuchat.txt",
            "chunks": 5
        },
        {
            "id": 2,
            "filename": "rag_notes.txt",
            "chunks": 8
        }
    ]
}

In [22]:
@tool
def mock_db_query(table: str, limit: int = 5) -> str:
    """Query the mock database and return rows from a table."""

    if table not in MOCK_DB:
        return f"Error: unknown table '{table}'."

    rows = MOCK_DB[table][:limit]

    return str(rows)

In [23]:
result = mock_db_query.invoke(
    {
        "table": "users",
        "limit": 2
    }
)

print(result)

[{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]


In [24]:
class AgentState(TypedDict):
    question: str
    route: str
    context: str
    tool_result: str
    answer: str

In [25]:
def router(state: AgentState):
    question = state["question"].lower()

    rag_keywords = [
        "docuchat",
        "rag",
        "document",
        "embedding",
        "retrieval"
    ]

    tool_keywords = [
        "calculate",
        "multiply",
        "divide",
        "database",
        "users"
    ]

    if any(keyword in question for keyword in rag_keywords):
        route = "rag"

    elif any(keyword in question for keyword in tool_keywords):
        route = "tool"

    else:
        route = "rag"

    return {
        "route": route
    }

In [26]:
def tool_node(state: AgentState):
    question = state["question"].lower()

    if "calculate" in question:
        expression = (
            question
            .replace("calculate", "")
            .strip()
        )

        result = calculator.invoke(
            {
                "expression": expression
            }
        )

    elif "users" in question:
        result = mock_db_query.invoke(
            {
                "table": "users",
                "limit": 5
            }
        )

    elif "database" in question:
        result = mock_db_query.invoke(
            {
                "table": "documents",
                "limit": 5
            }
        )

    else:
        result = "No suitable tool found."

    return {
        "tool_result": result
    }

In [27]:
def responder(state: AgentState):
    question = state["question"]
    route = state["route"]

    if route == "rag":
        information = state.get("context", "")

        prompt = f"""
Answer the user's question using only the provided context.

Question:
{question}

Context:
{information}

If the context does not contain enough information, say that
the information was not found in the available documents.
"""

    else:
        information = state.get("tool_result", "")

        prompt = f"""
Answer the user's question using the tool result.

Question:
{question}

Tool result:
{information}
"""

    response = model.invoke(prompt)

    return {
        "answer": response.content
    }

In [28]:
def route_function(state: AgentState):
    return state["route"]

In [29]:
builder = StateGraph(AgentState)

builder.add_node("router", router)
builder.add_node("rag", rag_node)
builder.add_node("tool", tool_node)
builder.add_node("responder", responder)

builder.add_edge(START, "router")

builder.add_conditional_edges(
    "router",
    route_function,
    {
        "rag": "rag",
        "tool": "tool"
    }
)

builder.add_edge("rag", "responder")
builder.add_edge("tool", "responder")

builder.add_edge("responder", END)

print("Graph structure created successfully.")

Graph structure created successfully.


In [30]:
result = graph.invoke(
    {
        "question": "Calculate 25 * 47",
        "route": "",
        "context": "",
        "tool_result": "",
        "answer": ""
    }
)

print("Route:", result["route"])

print("\nTool result:")
print(result["tool_result"])

print("\nAnswer:")
print(result["answer"])

Route: tool

Tool result:
1175

Answer:
The result of \(25 \times 47\) is **1175**.


In [31]:
result = graph.invoke(
    {
        "question": "Show me the users in the database.",
        "route": "",
        "context": "",
        "tool_result": "",
        "answer": ""
    }
)

print("Route:", result["route"])

print("\nTool result:")
print(result["tool_result"])

print("\nAnswer:")
print(result["answer"])

Route: tool

Tool result:
[{'id': 1, 'name': 'Arun', 'email': 'arun@example.com'}, {'id': 2, 'name': 'Raj', 'email': 'raj@example.com'}]

Answer:
Here are the users currently stored in the database:

| ID | Name | Email |
|----|------|-------------------|
| 1  | Arun | arun@example.com |
| 2  | Raj  | raj@example.com |

If you need more details or want to perform any other operation, just let me know!


In [32]:
checkpointer = InMemorySaver()

graph_with_memory = builder.compile(
    checkpointer=checkpointer
)

print("Graph compiled with checkpointer.")

Graph compiled with checkpointer.


In [33]:
config = {
    "configurable": {
        "thread_id": "demo-001"
    }
}

result = graph_with_memory.invoke(
    {
        "question": "What is RAG?",
        "route": "",
        "context": "",
        "tool_result": "",
        "answer": ""
    },
    config=config
)

print("Answer:")
print(result["answer"])

Answer:
RAG stands for **Retrieval‑Augmented Generation**. It is a technique that combines document retrieval with language‑model generation: relevant context is retrieved first, and then the language model uses that context to generate an answer.


In [34]:
state_snapshot = graph_with_memory.get_state(config)

print("Saved state:")
print(state_snapshot.values)

Saved state:
{'question': 'What is RAG?', 'route': 'rag', 'context': 'Retrieval-Augmented Generation combines document retrieval\n    with language model generation. Relevant context is retrieved\n    before the answer is generated.', 'tool_result': '', 'answer': 'RAG stands for **Retrieval‑Augmented Generation**. It is a technique that combines document retrieval with language‑model generation: relevant context is retrieved first, and then the language model uses that context to generate an answer.'}


In [36]:
import langgraph.types as lg_types

print("LangGraph version:", langgraph.__version__)

print("\nInterrupt-related APIs:")

print([
    name
    for name in dir(lg_types)
    if "interrupt" in name.lower()
])

NameError: name 'langgraph' is not defined

In [37]:
from langgraph.types import interrupt

print("interrupt imported successfully.")

interrupt imported successfully.


In [38]:
def approval_node(state: AgentState):
    decision = interrupt(
        {
            "type": "human_approval",
            "question": state["question"],
            "message": "Approve this tool action?"
        }
    )

    if decision == "approved":
        return {
            "tool_result": "Human approved the tool action."
        }

    return {
        "tool_result": "Human rejected the tool action."
    }

In [39]:
approval_builder = StateGraph(AgentState)

approval_builder.add_node("router", router)
approval_builder.add_node("approval", approval_node)
approval_builder.add_node("tool", tool_node)
approval_builder.add_node("rag", rag_node)
approval_builder.add_node("responder", responder)

approval_builder.add_edge(
    START,
    "router"
)

approval_builder.add_conditional_edges(
    "router",
    route_function,
    {
        "rag": "rag",
        "tool": "approval"
    }
)

approval_builder.add_edge(
    "approval",
    "tool"
)

approval_builder.add_edge(
    "rag",
    "responder"
)

approval_builder.add_edge(
    "tool",
    "responder"
)

approval_builder.add_edge(
    "responder",
    END
)

print("Approval workflow created successfully.")

Approval workflow created successfully.


In [40]:
approval_checkpointer = InMemorySaver()

approval_graph = approval_builder.compile(
    checkpointer=approval_checkpointer
)

print("Approval graph compiled successfully.")

Approval graph compiled successfully.


In [41]:
mermaid_diagram = approval_graph.get_graph().draw_mermaid()

print(mermaid_diagram)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router(router)
	approval(approval)
	tool(tool)
	rag(rag)
	responder(responder)
	__end__([<p>__end__</p>]):::last
	__start__ --> router;
	approval --> tool;
	rag --> responder;
	router -. &nbsp;tool&nbsp; .-> approval;
	router -.-> rag;
	tool --> responder;
	responder --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [42]:
approval_config = {
    "configurable": {
        "thread_id": "approval-demo-001"
    }
}

result = approval_graph.invoke(
    {
        "question": "Calculate 25 * 47",
        "route": "",
        "context": "",
        "tool_result": "",
        "answer": ""
    },
    config=approval_config
)

print(result)

{'question': 'Calculate 25 * 47', 'route': 'tool', 'context': '', 'tool_result': '', 'answer': '', '__interrupt__': [Interrupt(value={'type': 'human_approval', 'question': 'Calculate 25 * 47', 'message': 'Approve this tool action?'}, id='b980baf71cce19ab06d5b174b96525d9', response_schema=None)]}
